# Netra ANPR — training on Google Colab

This notebook trains both deep learning models on Colab's free GPU and gives you
the weights to use with the dashboard on your own computer.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Total time on a T4: roughly 15–25 minutes for the detector and 25–40 minutes for the character reader.

## 1. Get the code
Replace `katkarvismaya19-web` with your GitHub username after you push the repository.

In [ ]:
!git clone https://github.com/katkarvismaya19-web/netra-anpr.git
%cd netra-anpr
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Kaggle access
On kaggle.com open **Settings → API → Create New Token**. This downloads `kaggle.json`.
Run the cell below and upload that file when asked.

In [ ]:
from google.colab import files
import os, shutil
uploaded = files.upload()                     # choose kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

## 3. Download and prepare the detection dataset

In [ ]:
!python scripts/download_dataset.py
!python scripts/prepare_dataset.py

## 4. Train the plate detector (YOLOv8)

In [ ]:
!python scripts/train_detector.py --device 0

In [ ]:
from IPython.display import Image, display
display(Image('outputs/detector/train/results.png', width=900))
display(Image('outputs/detector/train/val_batch0_pred.jpg', width=900))

## 5. Generate synthetic plates and train the character reader (CRNN)

In [ ]:
!python scripts/generate_plates.py --preview 14
display(Image('outputs/synthetic_preview.png'))
!python scripts/generate_plates.py

In [ ]:
!python scripts/train_ocr.py --workers 2

In [ ]:
display(Image('outputs/ocr/training_curves.png', width=900))

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py --detector --ocr data/ocr/synthetic/val

## 7. Download the trained weights
This zips `models/` and `outputs/` (training curves, metrics). Extract the zip in your
local project folder so that `models/plate_detector.pt` and `models/crnn_plate.pt` exist,
then run `streamlit run app/main.py`.

In [ ]:
!zip -qr netra_trained.zip models outputs -x "outputs/detector/train/weights/*"
files.download('netra_trained.zip')